# DepthScape experiment 0001: relative-depth baseline

This notebook runs the pinned Depth Anything V2 Small checkpoint on one landscape image. The result is **unitless relative proximity**, not metric depth: white/larger values are nearer.

> Colab is a remote runtime. An image uploaded here is transferred to your temporary Colab VM. Use the generated demo image for the first run, and do not upload sensitive media.

In [ ]:
from pathlib import Path
import base64
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/doyeon-playground/depth-scape.git"
REPO_DIR = Path("/content/depth-scape")

def private_github_environment():
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        return None
    if not token:
        return None
    credential = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    environment = os.environ.copy()
    environment.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {credential}",
    })
    return environment

if not (REPO_DIR / ".git").exists():
    clone = subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPO_DIR)],
        env=private_github_environment(),
    )
    if clone.returncode != 0:
        raise RuntimeError("For a private repository, add a read-only fine-grained GitHub token to Colab Secrets as GITHUB_TOKEN. Never paste it into this notebook.")
else:
    print(f"Reusing {REPO_DIR}; use Runtime > Disconnect and delete runtime for a fresh checkout.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[depth]"], check=True)
project_revision = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("DepthScape revision:", project_revision)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch installation failed. Restart the runtime and rerun this cell.")

In [ ]:
from IPython.display import display
from PIL import Image

sample_path = Path("/content/demo-landscape.png")
subprocess.run([sys.executable, str(REPO_DIR / "samples/create_demo_landscape.py"), str(sample_path)], check=True)
display(Image.open(sample_path))

In [ ]:
output_dir = Path("/content/depth-run")
subprocess.run(
    [
        sys.executable,
        "-m",
        "depth_scape.cli",
        str(sample_path),
        "--output-dir",
        str(output_dir),
        "--device",
        "auto",
        "--overwrite",
    ],
    check=True,
)

In [ ]:
import json

manifest = json.loads((output_dir / "run.json").read_text(encoding="utf-8"))
display(Image.open(output_dir / "depth-preview.png"))
{
    "projectRevision": project_revision,
    "modelRevision": manifest["model"]["revision"],
    "inputDimensions": manifest["source"]["normalizedDimensions"],
    "device": manifest["configuration"]["device"],
    "precision": manifest["configuration"]["precision"],
    "modelLoadSeconds": manifest["performance"]["modelLoadSeconds"],
    "inferenceSeconds": manifest["performance"]["inferenceSeconds"],
    "peakAcceleratorMemoryBytes": manifest["performance"]["peakAcceleratorMemoryBytes"],
    "packages": manifest["software"]["packages"],
}

## Optional: try your own non-sensitive image

Set `USE_OWN_IMAGE` to `True`. The file is uploaded to the temporary Colab runtime, validated by the same JPG/PNG contract, and written to `/content/own-depth-run`.

In [ ]:
USE_OWN_IMAGE = False

if USE_OWN_IMAGE:
    from google.colab import files

    uploaded = files.upload()
    own_path = Path("/content") / next(iter(uploaded))
    own_output = Path("/content/own-depth-run")
    subprocess.run(
        [sys.executable, "-m", "depth_scape.cli", str(own_path), "--output-dir", str(own_output), "--overwrite"],
        check=True,
    )
    display(Image.open(own_output / "depth-preview.png"))
else:
    print("Set USE_OWN_IMAGE = True to upload a JPG or PNG.")